In [1]:
"""
Notebook: spark_data_preprocessing.ipynb

Objetivo:
Simular un pipeline de preprocesamiento escalable usando Apache Spark
sobre el dataset original BRFSS 2015 (≈440k registros, 330 variables),
tal y como se haría en un entorno corporativo con grandes volúmenes de datos.

Este notebook se centra exclusivamente en la ingestión, validación y
limpieza inicial de los datos.

Autor: Jesús Rodríguez
Fecha: 22/01/2026
"""

'\nNotebook: spark_data_preprocessing.ipynb\n\nObjetivo:\nSimular un pipeline de preprocesamiento escalable usando Apache Spark\nsobre el dataset original BRFSS 2015 (≈440k registros, 330 variables),\ntal y como se haría en un entorno corporativo con grandes volúmenes de datos.\n\nEste notebook se centra exclusivamente en la ingestión, validación y\nlimpieza inicial de los datos.\n\nAutor: Jesús Rodríguez\nFecha: 22/01/2026\n'

## 1. Configuración inicial

In [2]:
# Librerías básicas
from pathlib import Path

# Spark (motor de procesamiento distribuido)
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    isnan,
    when,
    round as spark_round,
)
from pyspark.sql.types import DoubleType

# Google Drive (solo necesario si se ejecuta en Google Colab)
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [3]:
# Inicialización de la sesión Spark
# Spark actúa como motor principal de procesamiento en este notebook
spark = (
    SparkSession.builder
    .appName("BRFSS2015_Spark_Preprocessing")
    .getOrCreate()
)

In [4]:
# Definición del nombre del archivo de datos
DATA_FILE = "raw_dataset.csv"

# Definición del directorio de datos según el entorno de ejecución
if IN_COLAB:
    DATA_DIR = Path("/content/drive/MyDrive/TFM/data")
else:
    DATA_DIR = Path("data")

# Ruta completa al archivo de datos
DATA_PATH = DATA_DIR / DATA_FILE

In [5]:
# Montaje de Google Drive (solo en Colab)
if IN_COLAB and not DATA_DIR.exists():
    drive.mount("/content/drive")

# Verificación de la existencia del archivo de datos
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo '{DATA_FILE}'. "
        f"Esperado en: {DATA_PATH}"
    )

In [6]:
# Carga del dataset original utilizando Spark
# - inferSchema=True infiere tipos automáticamente
# - header=True considera la primera fila como nombres de columnas
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_PATH))
)

In [7]:
if IN_COLAB:
    # Entorno de recursos limitado → pocas particiones para no sobrecargar memoria
    num_partitions = 2
else:
    # Producción / cluster → usar paralelismo por defecto del cluster
    num_partitions = spark.sparkContext.defaultParallelism

# Reparticionamos el DataFrame para optimizar operaciones distribuidas
df_raw = df_raw.repartition(num_partitions)

# Verificación del número de particiones
print(f"Número de particiones: {df_raw.rdd.getNumPartitions()}")

Número de particiones: 2


In [8]:
# Información general del dataset
print(f"Número de registros: {df_raw.count()}")
print(f"Número de variables: {len(df_raw.columns)}")

Número de registros: 441456
Número de variables: 330


In [9]:
# Estructura del dataset y tipos de datos
df_raw.printSchema()

root
 |-- _STATE: double (nullable = true)
 |-- FMONTH: double (nullable = true)
 |-- IDATE: integer (nullable = true)
 |-- IMONTH: integer (nullable = true)
 |-- IDAY: integer (nullable = true)
 |-- IYEAR: integer (nullable = true)
 |-- DISPCODE: double (nullable = true)
 |-- SEQNO: double (nullable = true)
 |-- _PSU: double (nullable = true)
 |-- CTELENUM: double (nullable = true)
 |-- PVTRESD1: double (nullable = true)
 |-- COLGHOUS: double (nullable = true)
 |-- STATERES: double (nullable = true)
 |-- CELLFON3: double (nullable = true)
 |-- LADULT: double (nullable = true)
 |-- NUMADULT: double (nullable = true)
 |-- NUMMEN: double (nullable = true)
 |-- NUMWOMEN: double (nullable = true)
 |-- CTELNUM1: double (nullable = true)
 |-- CELLFON2: double (nullable = true)
 |-- CADULT: double (nullable = true)
 |-- PVTRESD2: double (nullable = true)
 |-- CCLGHOUS: double (nullable = true)
 |-- CSTATE: double (nullable = true)
 |-- LANDLINE: double (nullable = true)
 |-- HHADULT: double (

In [10]:
# Visualización de las primeras filas del dataset
df_raw.show(5, truncate=False)

+------+------+--------+------+----+-----+--------+-------------+-------------+--------+--------+--------+--------+--------+------+--------+------+--------+--------+--------+------+--------+--------+------+--------+-------+-------+--------+--------+--------+--------+--------+-------+--------+-------+------+--------+-------+-------+--------+--------+--------+-------+-------+--------+--------+--------+--------+--------+--------+--------+--------+---+-------+-----+--------+--------+--------+-------+--------+-------+--------+-------+--------+-------+-------+--------+--------+--------+-----+------+--------+--------+--------+--------+--------+--------+--------+-------+-------+--------+--------+--------+--------+------+-------+-------+-------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+-------+--------+--------+--------+--------+-------+--------+--------+--------+--------+----

## 2. Análisis exploratorio inicial del target

In [11]:
# Distribución absoluta y relativa del target
target_distribution = (
    df_raw
    .groupBy("DIABETE3")
    .count()
    .withColumnRenamed("count", "Conteo")
)

# Cálculo del porcentaje sobre el total
total_count = df_raw.count()

target_distribution = (
    target_distribution
    .withColumn(
        "Porcentaje (%)",
        (target_distribution["Conteo"] / total_count) * 100
    )
    .orderBy("DIABETE3")
)

print("Distribución de clases del target:")
target_distribution.show(truncate=False)

Distribución de clases del target:
+--------+------+---------------------+
|DIABETE3|Conteo|Porcentaje (%)       |
+--------+------+---------------------+
|NULL    |7     |0.0015856619912290241|
|1.0     |57256 |12.969808995686998   |
|2.0     |3608  |0.8172954949077598   |
|3.0     |372104|84.29016708346924    |
|4.0     |7690  |1.741962958935885    |
|7.0     |598   |0.1354608386792795   |
|9.0     |193   |0.043718966329600234 |
+--------+------+---------------------+



## 3. Cribado de columnas

In [12]:
# Se conservan únicamente las clases relevantes del target
TARGET_CLASSES = [1, 3]

df_raw = df_raw.filter(df_raw["DIABETE3"].isin(TARGET_CLASSES))

### 3.1. Primer cribado de columnas (330 variables)

In [13]:
# Eliminación de registros duplicados
df_raw = df_raw.dropDuplicates()

In [14]:
# Definición de columnas irrelevantes para la detección de diabetes
COLS_TO_DROP_STAGE_1 = [
    '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO',
    '_PSU', 'QSTVER', 'QSTLANG', 'MSCODE', '_STSTR', '_STRWT', '_RAWRAKE',
    '_WT2RAKE', '_CLLCPWT', '_DUALUSE', '_DUALCOR', '_LLCPWT', 'CTELENUM',
    'CELLFON3', 'CTELNUM1', 'CELLFON2', 'CADULT', 'CCLGHOUS', 'CSTATE',
    'LANDLINE', 'NUMPHON2', 'CPDEMO1', 'INTERNET', 'PVTRESD1', 'COLGHOUS',
    'STATERES', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'PVTRESD2',
    'HHADULT', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'RENTHOM1',
    'NUMHHOL2', 'CHILDREN', 'INCOME2', 'CDHELP', 'CDSOCIAL', '_CHLDCNT',
    '_INCOMG', 'DIABAGE2', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'CHCKIDNY',
    'PDIABTST', 'PREDIAB1', 'INSULIN', 'FEETCHK2', 'DOCTDIAB', 'CHKHEMO3',
    'FEETCHK', 'EYEEXAM', 'DIABEYE', 'DIABEDU', 'MARITAL', 'EDUCA','VETERAN3',
    'EMPLOY1', 'SEATBELT', 'CDDISCUS', 'SCNTMNY1', 'SCNTMEL1', 'SCNTPAID',
    'SCNTWRK1', 'SCNTLPAD', 'SCNTLWK1', 'SXORIENT', 'TRNSGNDR', 'RCSGENDR',
    'RCSRLTN2', 'EMTSUPRT', 'LSATISFY', 'ADPLEASR', 'ADDOWN', 'MISTMNT',
    '_EDUCAG', '_RFSEAT2', '_RFSEAT3', '_LMTWRK1', '_LMTSCL1', 'CAREGIV1',
    'CRGVREL1', 'CRGVLNG1', 'CRGVHRS1', 'CRGVPRB1', 'CRGVPERS', 'CRGVHOUS',
    'CRGVMST2', 'CRGVEXPT', 'CDHOUSE', 'CDASSIST', 'WEIGHT2', 'HEIGHT3',
    'HTIN4', 'HTM4', 'WTKG3', '_BMI5', '_RFBMI5', 'STOPSMK2', 'LASTSMK2',
    'USENOW3', '_SMOKER3', '_RFSMOK3', 'ALCDAY5', 'DRNK3GE5', 'MAXDRNKS',
    'DRNKANY5', 'DROCDY3_', '_RFBING5', '_DRNKWEK', '_RFDRHV5', 'FVGREEN',
    'ARTHEXER', 'FVORANG', 'VEGETAB1', 'GRENDAY_', 'ORNGDAY_', 'VEGEDA1_',
    '_MISVEGN', '_VEGLT1', '_VEG23', '_VEGETEX', 'FTJUDA1_', 'FRUTDA1_',
    'FRUITJU1', 'FRUIT1', '_MISFRTN', '_FRTLT1', '_FRT16', '_FRUITEX',
    'FVBEANS', 'EXERANY2', 'EXRACT11', 'EXRACT21', 'EXEROFT2', 'EXERHMM2',
    'ADMOVE', 'EXACTOT1', 'EXACTOT2', '_TOTINDA', 'METVL11_', 'METVL21_',
    'MAXVO2_', 'FC60_', 'ACTIN11_', 'ACTIN21_', 'PADUR1_', 'PADUR2_',
    'PAFREQ1_', 'PAFREQ2_', '_MINAC11', '_MINAC21', 'STRFREQ_', 'PAMIN11_',
    'PAMIN21_', 'PA1MIN_', 'PAVIG11_', 'PAVIG21_', 'PA1VIGM_', '_PA150R2',
    '_PA300R2', '_PA30021', '_PASTRNG', '_PAREC1', '_PASTAE1', '_LMTACT1',
    'LMTJOIN3', 'ARTHSOCL', 'JOINPAIN', 'PAINACT2', 'TOLDHI2', 'QLMENTL2',
    'QLSTRES2', 'QLHLTH2', '_RFHLTH', 'VIDFCLT2', 'VIREDIF3', 'VIPRFVS2',
    'VINOCRE2', 'VIEYEXM2', 'VIINSUR2', 'VICTRCT4', 'VIGLUMA2', 'VIMACDG2',
    'ASTHMAGE', 'ASATTACK', 'ASERVIST', 'ASDRVIST', 'ASRCHKUP', 'ASACTLIM',
    'ASYMPTOM', 'ASNOSLEP', 'ASTHMED3', 'ASINHALR', 'CASTHDX2', 'CASTHNO2',
    '_LTASTH1', '_CASTHM1', '_ASTHMS1', 'HAREHAB1', 'STREHAB1', 'CVDASPRN',
    'ASPUNSAF', '_MICHD', 'RLIVPAIN', 'RDUCHART', 'RDUCSTRK', 'ARTTODAY',
    'ARTHWGT', 'ARTHEDU', '_DRDXAR1', '_RFCHOL',  '_CHISPNC', '_CRACE1',
    '_CPRACE', '_PRACE1', '_MRACE1', '_HISPANC', '_RACEG21', '_RACEGR3',
    '_RACE_G1', '_AGE65YR', '_AGE80', '_AGE_G', '_RFHYPE5', 'CHOLCHK',
    'HIVTST6', 'HIVTSTD3', 'WHRTST10', 'HADMAM', 'HOWLONG', 'HADPAP2',
    'LASTPAP2', 'HPVTEST', 'HPLSTTST', 'HADHYST2', 'PROFEXAM', 'LENGEXAM',
    'BLDSTOOL', 'LSTBLDS3', 'HADSIGM3', 'HADSGCO1', 'LASTSIG3', 'PSATEST1',
    'PSATIME', 'PCPSARS1', 'PCPSADE1', 'PCDMDECN', '_AIDTST3', '_CHOLCHK',
    'FLUSHOT6', 'FLSHTMY2', 'IMFVPLAC', 'PNEUVAC3', 'TETANUS', 'HPVADVC2',
    'HPVADSHT', 'SHINGLE2', '_FLSHOT6', '_PNEUMO2', 'DRADVISE', 'PREGNANT',
    'PCPSAAD2', 'PCPSADI1', 'PCPSARE1', '_HCVU651', 'STRENGTH', 'PAMISS1_',
    'SMOKDAY2', 'EXERHMM1', 'WTCHSALT', 'LONGWTCH', 'BEANDAY_', '_FRTRESP',
    '_VEGRESP', '_PAINDX1', 'CIMEMLOS', 'ASTHMA3', 'ASTHNOW', 'CHCSCNCR',
    'CHCOCNCR', 'CHCCOPD1', 'BLDSUGAR'
]

In [15]:
# Eliminación de columnas irrelevantes
# El operador * desempaqueta la lista y pasa cada nombre de columna
df_cleaned_1 = df_raw.drop(*COLS_TO_DROP_STAGE_1)

In [16]:
# Comprobación de consistencia del número total de columnas
total_columns = len(df_cleaned_1.columns) + len(COLS_TO_DROP_STAGE_1)

print(f"Total de columnas consideradas en el cribado: {total_columns}")

Total de columnas consideradas en el cribado: 330


### 3.2. Segundo cribado de columnas (34 variables)

In [17]:
# Crear lista de expresiones de conteo de nulos por columna
nulls_expr = [
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    for c in df_cleaned_1.columns
]

# Ejecutar conteo distribuido
nulls_df = df_cleaned_1.select(nulls_expr)

# Transformar el resultado en DataFrame de Spark con formato "columna, nulos"
# - Primero un melt manual usando selectExpr y stack
MELT_EXPR = ", ".join(
    [f"'{c}', {c}" for c in df_cleaned_1.columns]
)
nulls_melted = nulls_df.selectExpr(
    f"stack({len(df_cleaned_1.columns)}, {MELT_EXPR}) as (Columna, Nulos)"
)

# Añadir columna de porcentaje de nulos
total_rows = df_cleaned_1.count()
nulls_melted = nulls_melted.withColumn(
    "Porcentaje (%)",
    spark_round((col("Nulos") / total_rows) * 100, 2)
)

# Ordenar por número de nulos descendente
nulls_melted = nulls_melted.orderBy(col("Nulos").desc())

# Mostrar tabla resumida de nulos
nulls_melted.show(truncate=False)

+--------+------+--------------+
|Columna |Nulos |Porcentaje (%)|
+--------+------+--------------+
|ADANXEV |409608|95.4          |
|ADTHINK |409571|95.39         |
|ADFAIL  |409557|95.39         |
|ADEAT1  |409550|95.39         |
|ADENERGY|409539|95.38         |
|ADSLEEP |409534|95.38         |
|ARTHDIS2|297604|69.31         |
|BPMEDS  |257357|59.94         |
|AVEDRNK2|223367|52.02         |
|POORHLTH|209652|48.83         |
|EXEROFT1|142903|33.28         |
|_VEGESUM|49754 |11.59         |
|_FRUTSUM|42573 |9.92          |
|_BMI5CAT|35155 |8.19          |
|SMOKE100|13903 |3.24          |
|DIFFALON|12998 |3.03          |
|DIFFDRES|12415 |2.89          |
|DIFFWALK|12030 |2.8           |
|DECIDE  |11456 |2.67          |
|BLIND   |10887 |2.54          |
+--------+------+--------------+
only showing top 20 rows


In [18]:
# Columnas con elevado número de valores nulos o menor relevancia clínica
COLS_TO_DROP_STAGE_2 = [
    'ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL',
    'ADTHINK', 'ADANXEV', 'ARTHDIS2',
    'POORHLTH', 'AVEDRNK2'
]

# Eliminación de columnas seleccionadas
# El operador * desempaqueta la lista de columnas para que drop() las reciba
# como argumentos individuales
df_cleaned_2 = df_cleaned_1.drop(*COLS_TO_DROP_STAGE_2)

# Comprobación de consistencia del número total de columnas
total_columns = len(df_cleaned_2.columns) + len(COLS_TO_DROP_STAGE_2)
print(f"Total de columnas consideradas en el cribado: {total_columns}")

Total de columnas consideradas en el cribado: 34


### 3.3. Tercer cribado de columnas (25 variables)

In [19]:
class MissingValueAnalyzerSpark:
    """
    Analiza porcentajes de valores especiales en un DataFrame de Spark
    de manera completamente distribuida.

    - No se trae nada al driver hasta que se haga show().
    - Mantiene operaciones paralelas y escalables para grandes datasets.
    """

    def __init__(self, df):
        """
        Inicializa con un DataFrame de Spark.

        Args:
            df: Spark DataFrame a analizar
        """
        self.df = df

    def calculate_percentage(self, columns: list, special_values: list):
        """
        Calcula el porcentaje de valores especiales por columna usando solo Spark.

        Args:
            columns: Columnas a analizar
            special_values: Valores considerados especiales

        Returns:
            Spark DataFrame con columnas:
            'Columna' y 'Porcentaje (%)'
        """
        total_rows_local = self.df.count()  # filas de manera distribuida

        # Expresiones SQL para cada columna
        exprs = [
            f"count(CASE WHEN {c} IN ({','.join(map(str, special_values))}) "
            f"THEN 1 END) / {total_rows_local} * 100 AS `{c}`"
            for c in columns
        ]

        # Fila con los porcentajes
        percent_row = self.df.selectExpr(exprs).limit(1)

        # Formato "columna, porcentaje" usando stack
        stack_expr = ", ".join([f"'{c}', `{c}`" for c in columns])
        stack_call = (
            f"stack({len(columns)}, "
            + f"{stack_expr}) "
            + "as (Columna, `Porcentaje (%)`)"
        )

        result_df = percent_row.selectExpr(stack_call)

        return result_df

    def calculate_all_groups(self, column_groups: dict):
        """
        Calcula porcentajes de valores especiales para varios grupos de columnas
        usando solo Spark.

        Args:
            column_groups: dict {nombre_grupo: (columnas, valores_especiales)}

        Returns:
            dict con Spark DataFrames como valores para cada grupo,
            listos para show() en notebook.
        """
        return {
            group_name: self.calculate_percentage(columns, special_values)
            for group_name, (columns, special_values) in column_groups.items()
        }

In [20]:
# Creación del analizador de valores especiales
missing_value_analyzer = MissingValueAnalyzerSpark(df_cleaned_2)

In [21]:
# Definición de grupos de columnas y valores especiales
COLUMN_GROUPS = {
    '7_9': (
        ['GENHLTH', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3',
         'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK',
         'DIFFDRES', 'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_PACAT1'],
        [7, 9]
    ),
    '77_88_99': (
        ['PHYSHLTH', 'MENTHLTH'],
        [77, 88, 99]
    ),
    '777_999': (
        ['EXEROFT1'],
        [777, 999]
    ),
    '9': (
        ['_RACE'],
        [9]
    ),
    '14': (
        ['_AGEG5YR'],
        [14]
    )
}

In [22]:
# Cálculo de resultados por grupo
missing_value_results = missing_value_analyzer.calculate_all_groups(COLUMN_GROUPS)

# Visualización de resultados
for group_name, df_result in missing_value_results.items():
    print(f"\n=== Resultados para grupo {group_name} ===")
    print(df_result)


=== Resultados para grupo 7_9 ===
DataFrame[Columna: string, Porcentaje (%): double]

=== Resultados para grupo 77_88_99 ===
DataFrame[Columna: string, Porcentaje (%): double]

=== Resultados para grupo 777_999 ===
DataFrame[Columna: string, Porcentaje (%): double]

=== Resultados para grupo 9 ===
DataFrame[Columna: string, Porcentaje (%): double]

=== Resultados para grupo 14 ===
DataFrame[Columna: string, Porcentaje (%): double]


In [23]:
# Columnas con valores especiales significativos que se eliminarán
COLS_TO_DROP_STAGE_3 = ['PHYSHLTH', 'MENTHLTH']

# Eliminación de columnas seleccionadas
df_cleaned_3 = df_cleaned_2.drop(*COLS_TO_DROP_STAGE_3)

In [24]:
# Comprobaciones finales de consistencia
total_columns = len(df_cleaned_3.columns) + len(COLS_TO_DROP_STAGE_3)
print(f"Total de columnas consideradas en el cribado: {total_columns}")
print(f"Número final de columnas: {len(df_cleaned_3.columns)}")

Total de columnas consideradas en el cribado: 25
Número final de columnas: 23


## 4. Transformación e imputación de variables

### 4.1. Variables que podrían estar relacionadas

#### BPMEDS y su relación con BPHIGH

In [25]:
# Indicador booleano auxiliar de valores nulos en BPMEDS
df_cleaned_3 = df_cleaned_3.withColumn(
    "BPMEDS_is_null", col("BPMEDS").isNull()
    )

In [26]:
# Distribución de nulos de BPMEDS según hipertensión diagnosticada (BPHIGH4)
# Cálculo del conteo por grupo
bphigh_distribution = (
    df_cleaned_3
    .groupBy("BPHIGH4", "BPMEDS_is_null")
    .agg(count("*").alias("Conteo"))
    .orderBy("BPHIGH4", "BPMEDS_is_null")
)

# Para ver la proporción por columna (BPHIGH4)
total_per_bphigh = (
    df_cleaned_3
    .groupBy("BPHIGH4")
    .agg(count("*").alias("Total"))
)

# Join para calcular proporciones
bphigh_distribution = bphigh_distribution.join(
    total_per_bphigh, on="BPHIGH4"
).withColumn(
    "Porcentaje", col("Conteo") / col("Total") * 100
)

print("Distribución de BPMEDS nulos según BPHIGH4:")
bphigh_distribution.show(truncate=False)

Distribución de BPMEDS nulos según BPHIGH4:
+-------+--------------+------+------+----------+
|BPHIGH4|BPMEDS_is_null|Conteo|Total |Porcentaje|
+-------+--------------+------+------+----------+
|7.0    |true          |816   |816   |100.0     |
|1.0    |false         |172003|172003|100.0     |
|4.0    |true          |4019  |4019  |100.0     |
|3.0    |true          |249170|249170|100.0     |
|2.0    |true          |2941  |2941  |100.0     |
|9.0    |true          |410   |410   |100.0     |
+-------+--------------+------+------+----------+



In [27]:
# Imputación clínica:
# Si no tiene hipertensión, no toma medicación → categoría 2 (NO)
df_cleaned_3 = df_cleaned_3.withColumn(
    "BPMEDS",
    when(
        col("BPMEDS").isNull(),
        2
    ).otherwise(
        col("BPMEDS")
    )
)

In [28]:
# Verificación de imputación en BPMEDS
# Se cuentan los registros donde BPMEDS sigue siendo nulo
remaining_nulls = (
    df_cleaned_3
    .filter(col("BPMEDS").isNull())
    .count()
)

print(f"Número de NaN restantes en BPMEDS: {remaining_nulls}")

Número de NaN restantes en BPMEDS: 0


In [29]:
# Eliminación de la variable auxiliar
df_cleaned_3 = df_cleaned_3.drop("BPMEDS_is_null")

#### EXEROFT1 y limitaciones de movilidad

In [30]:
# Indicador auxiliar de valores nulos en EXEROFT1
df_cleaned_3 = df_cleaned_3.withColumn(
    "EXEROFT1_is_null",
    col("EXEROFT1").isNull()
)

In [31]:
# Análisis de la relación entre nulos en EXEROFT1 y limitaciones funcionales
FUNCTIONAL_LIMIT_COLS = ["DIFFALON", "DIFFDRES", "DIFFWALK"]


for variable in FUNCTIONAL_LIMIT_COLS:
    # Conteo por combinación de variable funcional y nulos en EXEROFT1
    distribution = (
        df_cleaned_3
        .groupBy(variable, "EXEROFT1_is_null")
        .agg(count("*").alias("Conteo"))
        .orderBy(variable, "EXEROFT1_is_null")
    )

    # Total por categoría de la variable funcional
    totals = (
        df_cleaned_3
        .groupBy(variable)
        .agg(count("*").alias("Total"))
    )

    # Cálculo del porcentaje por columna (equivalente a normalize='columns')
    distribution = (
        distribution
        .join(totals, on=variable)
        .withColumn(
            "Porcentaje",
            col("Conteo") / col("Total") * 100
        )
    )

    print(f"\nDistribución de nulos de EXEROFT1 según {variable}:")
    distribution.show(truncate=False)


Distribución de nulos de EXEROFT1 según DIFFALON:
+--------+----------------+------+------+------------------+
|DIFFALON|EXEROFT1_is_null|Conteo|Total |Porcentaje        |
+--------+----------------+------+------+------------------+
|2.0     |true            |110307|382206|28.860614433054426|
|1.0     |false           |13975 |32361 |43.18469762986311 |
|2.0     |false           |271899|382206|71.13938556694558 |
|7.0     |true            |460   |877   |52.451539338654506|
|7.0     |false           |417   |877   |47.5484606613455  |
|9.0     |false           |166   |918   |18.082788671023962|
|1.0     |true            |18386 |32361 |56.81530237013689 |
|9.0     |true            |752   |918   |81.91721132897604 |
+--------+----------------+------+------+------------------+


Distribución de nulos de EXEROFT1 según DIFFDRES:
+--------+----------------+------+------+------------------+
|DIFFDRES|EXEROFT1_is_null|Conteo|Total |Porcentaje        |
+--------+----------------+------+------+--

In [32]:
# Conclusión clínica:
# Los NaN en EXEROFT1 no presentan un patrón funcional claro
# Se elimina la variable auxiliar
df_cleaned_3 = df_cleaned_3.drop("EXEROFT1_is_null")

### 4.2. Tratamiento de variables especiales

#### EXEROFT1 – decodificación de frecuencia de ejercicio

In [33]:
# Decodificación de la variable EXEROFT1
# - 101–199 → veces por semana
# - 201–299 → veces por mes (convertido a semana)
# - 777, 999 → No sabe / No contesta (se conservan)
df_cleaned_3 = df_cleaned_3.withColumn(
    "EXEROFT1",
    when(
        col("EXEROFT1").isNull(),
        None
    ).when(
        (col("EXEROFT1") >= 101) & (col("EXEROFT1") <= 199),
        col("EXEROFT1") - 100
    ).when(
        (col("EXEROFT1") >= 201) & (col("EXEROFT1") <= 299),
        (col("EXEROFT1") - 200) / (30.0 / 7.0)
    ).when(
        col("EXEROFT1").isin(777, 999),
        col("EXEROFT1")
    ).otherwise(
        None
    )
)

In [34]:
# Inspección de valores únicos tras la transformación
# Limita el número de valores para evitar recolectar grandes volúmenes
df_cleaned_3.select("EXEROFT1").distinct().show(20, truncate=False)

+------------------+
|EXEROFT1          |
+------------------+
|15.4              |
|1.4000000000000001|
|70.0              |
|8.0               |
|7.933333333333334 |
|0.4666666666666667|
|7.7               |
|4.433333333333334 |
|1.6333333333333333|
|0.9333333333333333|
|5.133333333333334 |
|9.333333333333334 |
|22.400000000000002|
|1.1666666666666667|
|4.9               |
|9.8               |
|6.066666666666666 |
|8.166666666666666 |
|16.333333333333332|
|11.666666666666668|
+------------------+
only showing top 20 rows


#### _FRUTSUM y _VEGESUM – normalización de consumo diario

In [35]:
# Conversión de raciones * 100 a raciones reales por día
df_cleaned_3 = (
    df_cleaned_3
    .withColumn("_FRUTSUM", col("_FRUTSUM") / 100)
    .withColumn("_VEGESUM", col("_VEGESUM") / 100)
)

In [36]:
# Verificación de valores transformados
# Valores únicos limitados para evitar recolectar grandes volúmenes
print("Valores únicos de _FRUTSUM tras la conversión:")
df_cleaned_3.select(
    spark_round(col("_FRUTSUM"), 2).alias("_FRUTSUM")
).distinct().show(20, truncate=False)

print("Valores únicos de _VEGESUM tras la conversión:")
df_cleaned_3.select(
    spark_round(col("_VEGESUM"), 2).alias("_VEGESUM")
).distinct().show(20, truncate=False)

Valores únicos de _FRUTSUM tras la conversión:
+--------+
|_FRUTSUM|
+--------+
|2.86    |
|0.66    |
|3.26    |
|5.86    |
|2.4     |
|0.07    |
|0.84    |
|24.03   |
|5.13    |
|1.41    |
|8.0     |
|2.41    |
|8.43    |
|0.87    |
|0.0     |
|6.17    |
|3.02    |
|1.42    |
|5.4     |
|4.23    |
+--------+
only showing top 20 rows
Valores únicos de _VEGESUM tras la conversión:
+--------+
|_VEGESUM|
+--------+
|2.86    |
|0.66    |
|3.26    |
|9.13    |
|15.5    |
|5.86    |
|184.0   |
|6.96    |
|1.82    |
|11.53   |
|4.19    |
|2.4     |
|0.07    |
|0.84    |
|9.72    |
|5.05    |
|2.41    |
|5.13    |
|22.83   |
|1.41    |
+--------+
only showing top 20 rows


### 4.3. Imputación

#### Eliminación inicial de NaN estructurales

In [37]:
# Eliminación de registros con valores nulos reales (estructurales)
# Spark elimina cualquier fila que contenga al menos un null
df_cleaned_3 = df_cleaned_3.dropna()

In [38]:
# Verificación básica del estado del dataset tras la limpieza
print(f"Número de registros: {df_cleaned_3.count()}")
print(f"Número de variables: {len(df_cleaned_3.columns)}")

# Esquema del DataFrame
df_cleaned_3.printSchema()

Número de registros: 257709
Número de variables: 23
root
 |-- GENHLTH: double (nullable = true)
 |-- BPHIGH4: double (nullable = true)
 |-- BPMEDS: double (nullable = true)
 |-- BLOODCHO: double (nullable = true)
 |-- HAVARTH3: double (nullable = true)
 |-- ADDEPEV2: double (nullable = true)
 |-- DIABETE3: double (nullable = true)
 |-- SEX: double (nullable = true)
 |-- QLACTLM2: double (nullable = true)
 |-- USEEQUIP: double (nullable = true)
 |-- BLIND: double (nullable = true)
 |-- DECIDE: double (nullable = true)
 |-- DIFFWALK: double (nullable = true)
 |-- DIFFDRES: double (nullable = true)
 |-- DIFFALON: double (nullable = true)
 |-- SMOKE100: double (nullable = true)
 |-- EXEROFT1: double (nullable = true)
 |-- _RACE: double (nullable = true)
 |-- _AGEG5YR: double (nullable = true)
 |-- _BMI5CAT: double (nullable = true)
 |-- _FRUTSUM: double (nullable = true)
 |-- _VEGESUM: double (nullable = true)
 |-- _PACAT1: double (nullable = true)



#### Identificación de valores especiales (“No sabe / Se negó”)

In [39]:
# Valores especiales que representan respuestas inválidas
# (_BMI5CAT, _FRUTSUM, _VEGESUM y SEX no presentan estas categorías)
# DIABETE3 ya fue filtrada previamente
INVALID_VALUES = {
    'GENHLTH': [7, 9],
    'BPHIGH4': [7, 9],
    'BPMEDS': [7, 9],
    'BLOODCHO': [7, 9],
    'HAVARTH3': [7, 9],
    'QLACTLM2': [7, 9],
    'USEEQUIP': [7, 9],
    'BLIND': [7, 9],
    'DECIDE': [7, 9],
    'DIFFWALK': [7, 9],
    'DIFFDRES': [7, 9],
    'DIFFALON': [7, 9],
    'SMOKE100': [7, 9],
    'ADDEPEV2': [7, 9],
    '_PACAT1': [9],
    '_RACE': [9],
    '_AGEG5YR': [14],
    'EXEROFT1': [777, 999],
}

In [40]:
# Sustitución de valores especiales por null (NaN en Spark)
for column, invalid_values in INVALID_VALUES.items():
    df_cleaned_3 = df_cleaned_3.withColumn(
        column,
        when(
            col(column).isin(invalid_values),
            None
        ).otherwise(
            col(column)
        )
    )

#### Separación por tipo de variable

In [41]:
# Definición de tipos de variables
CATEGORICAL_NOMINAL = [
    'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3', 'QLACTLM2',
    'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES',
    'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_RACE',
]

CATEGORICAL_ORDINAL = [
    'GENHLTH', '_PACAT1', '_AGEG5YR',
]

NUMERICAL = [
    'EXEROFT1',
]

#### Imputación

In [42]:
# Imputación de variables categóricas (nominales y ordinales)
df_cleaned_3 = df_cleaned_3.fillna(
    -1,
    subset=CATEGORICAL_NOMINAL + CATEGORICAL_ORDINAL,
)


In [43]:
# Imputación de variables numéricas mediante la mediana
for column in NUMERICAL:
    median_value = df_cleaned_3.approxQuantile(
        column,
        [0.5],
        0.0,
    )[0]

    df_cleaned_3 = df_cleaned_3.withColumn(
        column,
        when(
            col(column).isNull(),
            median_value,
        ).otherwise(
            col(column).cast(DoubleType())
        ),
    )

## 5. Exportación

In [44]:
# Verificación final del esquema y tipos de datos
df_cleaned_3.printSchema()

# Verificación básica del número de registros y columnas
print(f"Número de filas: {df_cleaned_3.count()}")
print(f"Número de columnas: {len(df_cleaned_3.columns)}")

root
 |-- GENHLTH: double (nullable = false)
 |-- BPHIGH4: double (nullable = false)
 |-- BPMEDS: double (nullable = false)
 |-- BLOODCHO: double (nullable = false)
 |-- HAVARTH3: double (nullable = false)
 |-- ADDEPEV2: double (nullable = false)
 |-- DIABETE3: double (nullable = true)
 |-- SEX: double (nullable = true)
 |-- QLACTLM2: double (nullable = false)
 |-- USEEQUIP: double (nullable = false)
 |-- BLIND: double (nullable = false)
 |-- DECIDE: double (nullable = false)
 |-- DIFFWALK: double (nullable = false)
 |-- DIFFDRES: double (nullable = false)
 |-- DIFFALON: double (nullable = false)
 |-- SMOKE100: double (nullable = false)
 |-- EXEROFT1: double (nullable = true)
 |-- _RACE: double (nullable = false)
 |-- _AGEG5YR: double (nullable = false)
 |-- _BMI5CAT: double (nullable = true)
 |-- _FRUTSUM: double (nullable = true)
 |-- _VEGESUM: double (nullable = true)
 |-- _PACAT1: double (nullable = false)

Número de filas: 257709
Número de columnas: 23


In [45]:
# Ruta de salida
OUTPUT_PATH = "/content/drive/MyDrive/TFM/data/cleaned_dataset_spark.csv"

# Exportación del dataset limpio a CSV
# - header=True incluye nombres de columnas
# - mode='overwrite' sobrescribe si existe
# - sep=',' para compatibilidad con CSV estándar
df_cleaned_3.write.csv(
    path=OUTPUT_PATH,
    mode='overwrite',
    header=True,
    sep=','
)